# `models/nafnet/`：NAFNet Student 模組手冊

## 1. 角色與檔案內容

本資料夾負責把 NAFNet 接成 KD pipeline 中可訓練的 **single-frame Student**。正式 KD training 的 Student 是 NAFNet，不是 `DemoStudent`。Teacher 接收完整 burst；NAFNet 只接收同一筆樣本的第一張 noisy reference frame。

| 檔案 | 責任 |
|---|---|
| `nafnet_arch.py` | 本專案內 NAFNet architecture，來自組員來源程式的 architecture implementation |
| `NAFNet_UPSTREAM_LICENSE.txt` | 隨 architecture 保留的上游授權文字 |
| `nafnet_student.py` | `NAFNetStudent` wrapper、輸入檢查、checkpoint 載入及 `create_student` factory |
| `demo_student.py` | 小型 placeholder CNN，只供 smoke/regression test，不是 NAFNet |
| `__init__.py` | Student package 的匯出介面 |
| `README.ipynb` | 本模組說明 |

正式執行時不需要在磁碟另放完整的 `NAFNet (from gpmate)` source folder；架構已放在本資料夾內。原本組員的 NAFNet source folder 不應因這個整合流程而被修改。

## 2. NAFNet 在 KD 中的 I/O 契約

Dataset 的 `reference_rgb` 是第一幀 noisy RGB，shape `[B, 3, H, W]`。`NAFNetStudent.forward()` 接收該張量並輸出相同 shape 的 RGB restoration：

```text
noisy reference RGB [B, 3, H, W]
       └─ NAFNetStudent / NAFNet ─→ student_output [B, 3, H, W]
```

輸出會與同一筆 clean reference 計算 Ground-truth loss，也會與 Burstormer 的 3-channel output 計算 output-level KD loss。此契約要求 Teacher output、Student output 和 target 對應相同 spatial crop、frame、channel order 及數值尺度。

`nafnet_student.py` 不會偷偷 clamp output 或自行正規化資料；前處理和值域屬於 Dataset / training protocol。正式實驗前要確認 GPMate NAFNet baseline、Teacher checkpoint 和 KD Dataset 使用一致的 RGB/linear-like domain 及尺度。

目前 NAFNet 使用組員 baseline 的 architecture config：RGB input/output、width 32、encoder blocks `[2, 2, 4, 8]`、middle 12、decoder `[2, 2, 2, 2]`。NAFNet 本身有 padding 對齊下採樣倍率並在輸出裁切回原始 H/W 的行為。這些資訊描述 architecture 接線，不表示正式訓練超參數已定案。

## 3. Factory、初始化與 checkpoint

正式 trainer 使用：

```text
models.nafnet.nafnet_student:create_student
```

Factory 的概念介面為 `create_student(device, checkpoint_path=None)`，回傳放在指定 device 的 `nn.Module`。未提供 checkpoint 時，NAFNet 從隨機初始化開始；若有 supervised NAFNet baseline checkpoint，可以用 `--student-checkpoint` 指定。Checkpoint 應是 state dict 或含 `model_state_dict` 的 dict，且架構設定需相符。

注意：Teacher 的 `weights/Burstormer.ckpt` 是預訓練 Teacher 權重，不是 NAFNet Student initialization。Student 的最佳 KD 權重由 `train_kd.py` 另存至 `checkpoints_kd/nafnet_kd_best.pt`。本機現有這個 checkpoint 是 9 張可用圖片、1 epoch CPU test 的產物，不是正式訓練結果。

## 4. 正式 Student 和 DemoStudent 的差別

| 情境 | 使用的 Student | 資料 | 目的 |
|---|---|---|---|
| `run_kd_smoke.py` regression | 可指定 `demo_student:create_student` | 隨機資料、placeholder sigma、toy target | 快速檢查 factories、shape、loss/backward/update |
| `run_kd_smoke.py` 真模型 integration | 可指定 `nafnet_student:create_student` | 仍是 smoke dummy dataset | 驗證 NAFNet 可接 smoke tensor，不代表訓練資料有效 |
| `train_kd.py` formal training entry | `nafnet_student:create_student` | Open Images source + on-the-fly burst/noise | train/validation loss 和 Student checkpoint |

`DemoStudent` 預設 learnable residual scale 初始為零，為 smoke plumbing 提供小型可訓練模型；它不具備 NAFNet 架構、不會有有意義的 denoising performance。若 smoke Student 使用 NAFNet factory，實際用的是 NAFNet，但 smoke data 仍是假資料，故 smoke 結果仍不是品質評估。

## 5. 本機驗證紀錄與解讀

目前已使用真實 NAFNet adapter，在 CPU 上完成：

1. NAFNet 建構與 forward shape 檢查。
2. 從同架構 state dict 載入 checkpoint 的測試。
3. 使用一筆實際下載 Open Images 合成樣本，搭配 Burstormer checkpoint 完成一次 Student 更新。
4. 使用 9 筆 usable source images 跑完 1 epoch，7 train／2 validation，保存 best checkpoint。

這證明 NAFNet 與目前 KD training loop 能執行；沒有證明一個 epoch 能訓練出可用 denoiser，也沒有比較 baseline、KD 或 Teacher 的 test scores。作者電腦只有 CPU，正式訓練需組員的 GPU 和相容 PyTorch/torchvision。

## 6. 與組員 baseline 的關係

組員 GPMate 專案中的 `nafnet.py`、`train.py`、`denoise_dataset.py` 和 `burst_utils.py` 是 NAFNet baseline/資料生成參考。KD 專案將 architecture 集中在此 folder，資料生成和 training loop 則放在 `data/` 與根目錄，不把整個 baseline trainer 複製進模型資料夾。

要做公平比較，不能只確認「兩邊用同一個 noise formula」；還需確認兩邊相同的 source split、crop/resize、frame 0 reference、gamma/white-level、parameter sampling、sample seeds、optimizer/epochs 等比較條件。現行 KD train sample access 會重新抽 crop/noise，尚未提供與 standalone baseline 逐筆同步的 manifest。

## 7. 交接時需保留的檔案

交付程式 ZIP 時保留 `nafnet_arch.py`、`nafnet_student.py`、`demo_student.py`、`__init__.py` 與 `NAFNet_UPSTREAM_LICENSE.txt`。不需要複製整個 GPMate folder，也不要把本機 `.venv`、raw dataset 或 `checkpoints_kd/` 當成模型 source 包進來。若組員需從現有 NAFNet baseline checkpoint 初始化，該 checkpoint 必須另外提供，並在執行時指定 `--student-checkpoint`。

## 8. 尚未完成

- 建立／交付可選的 supervised NAFNet baseline checkpoint（如實驗決定要用）。
- 確認 NAFNet 與 Burstormer output 對應相同值域、參考幀和後處理規則。
- 建立 baseline 與 KD 共用 sample manifest，若公平 protocol 要求逐樣本相同。
- 以固定 test set 實作 PSNR/SSIM 或其他 score 評估。

上述事項未定案前，本資料夾的 NAFNet integration 只表示「Student architecture 已接入並可訓練」，不表示完成模型效能研究。

## 9. Architecture wrapper 做什麼、不做什麼

`nafnet_student.py` 把本地 architecture 建成 `self.model`，在 forward 時先檢查輸入為 4D RGB tensor，再呼叫底層 NAFNet。底層 NAFNet 會依 encoder 階層數把 H/W padding 到可下採樣的倍數，經 encoder、middle blocks 和 decoder 後產生 residual output，並裁回原始 H/W。這讓常見任意尺寸輸入可維持 output spatial shape，但實際模型仍需符合顯存與 Teacher/target shape 要求。

Wrapper 不負責：

- 讀取 Open Images 或產生 noise（由 `data/` 負責）。
- 建立 Teacher、計算 loss、選 lambda 或更新 optimizer（由 trainer 負責）。
- 自動套用 display conversion、clamp、normalization 或 post-processing。
- 自行決定是否讀取 pretrained Student weights；預設隨機初始化。

這些邊界可避免把 Dataset 值域處理藏進模型 adapter，並讓 smoke 和正式資料路徑使用同一個 Student 接口。

## 10. Checkpoint 格式與常見問題

組員若提供 `nafnet_best.pt`，先確認 checkpoint 是 PyTorch 可讀的檔案，內含 `model_state_dict` 或本身就是 state dict，key/shape 對應本專案 `nafnet_arch.NAFNet` config。由 `--student-checkpoint` 傳給 trainer 時，factory 會載入至架構內層 model。架構 width 或 block numbers 不同會造成 `load_state_dict` key/shape error，不能透過 `strict=False` 隱藏不相容而假裝成功。

常見問題：

- `FileNotFoundError`：檢查 checkpoint 絕對路徑和檔名。
- `Missing key(s)`／`Unexpected key(s)`：檢查 checkpoint 是否為完整模型 state dict，是否存成 `module.` 前綴或不同 architecture。
- Tensor size mismatch：確認 NAFNet config 與該權重 training config 一致。
- 模型能載入但輸出範圍不同：load success 只證明參數 shape 對應，不保證 input normalization 和 output domain 一致。

本專案現有 334 MB `checkpoints_kd/nafnet_kd_best.pt` 是 KD run 的 training output，不應拿它當作外部 pretrained initialization 再重新跑，除非實驗明確定義為 resume/finetune。正式使用者需分辨 `--student-checkpoint`（初始化權重）和 trainer 自動輸出的 best KD checkpoint。

## 11. Trainer 如何使用 Student

`train_kd.py` 透過 factory 名稱載入 Student，將模型放在命令列指定裝置。Optimizer 只接收 `requires_grad=True` 的 Student parameter。每筆 batch 的 `reference_rgb` 喂給 NAFNet；輸出 shape 必須與 Teacher output 和 clean target 相同。Teacher 不會因 Student factory 選擇而改變，Student adapter 也不讀完整 burst。

Trainer 的預設訓練初始化為 from-scratch。若需要 supervised baseline 先訓練再 KD，必須把 baseline checkpoint 以 `--student-checkpoint` 指定，且先確認 architecture 和 data condition；README 不假設已有 baseline weights。

## 12. CPU、GPU 與效能預期

NAFNet 本身可在 CPU 執行，作者已用 CPU 完成 9 張可用圖片的 1 epoch run。`--device cpu` 適用於檢查流程；實際每張 128×128 的 Burstormer+NAFNet forward/backward 在大量樣本、許多 epochs 下會明顯慢，不建議用來估計正式訓練時間。

GPU 使用條件包括：

1. 電腦有支援的 NVIDIA GPU 與正常 driver。
2. PyTorch 和 torchvision 是匹配的 CUDA build。
3. `torch.cuda.is_available()` 回傳 True。
4. `--device cuda` 與實際環境相符。

如果 CUDA 初始化報 `Torch not compiled with CUDA enabled`，代表目前 PyTorch wheel 無 CUDA 支援，不是 NAFNet forward 寫死 CUDA。切回 CPU 可跑小型驗證；正式 GPU 訓練需在組員機器另裝 GPU 版 dependencies。

## 13. NAFNet 模組狀態表

| 類別 | 現況 |
|---|---|
| Architecture | 本地 `nafnet_arch.py` 已存在，可由 adapter 建構 |
| Student contract/factory | 已接入 `train_kd.py`，支援 device 和 optional checkpoint |
| Smoke placeholder | `demo_student.py` 保留，不會取代正式 NAFNet |
| Input Dataset | 使用 `data.nafnet_kd_dataset.NAFNetKDBurstDataset` |
| Baseline training script | GPMate source 的 `train.py` 不在本資料夾；此專案尚無獨立 baseline entry |
| KD training | `train_kd.py` 已完成 1 epoch CPU integration run |
| Test metrics | 尚無獨立 test/PSNR/SSIM evaluation |
| Checkpoint | 有小型試跑 artifact，不是正式 KD model |

對外傳送純程式 ZIP 時，architecture license 必須保留；raw dataset、`.venv` 和小型 training output 不屬於 NAFNet source module。
